# 6a: LSTM-GCN Rolling Pearson - Edge Perturbation Ablation

Ablation study to assess GCN robustness to edge perturbation.

**Configure at top:**
- `PERTURBATION_MODE`: "add" or "subtract"
- `PERTURBATION_FRACTION`: fraction of real edges to perturb

**Fixed:** threshold=0.5, lookback=20, equity returns

Run this notebook multiple times with different (mode, fraction) combinations
to produce the robustness sweep.

## 1. Setup

In [ ]:
!pip install -q tensorflow>=2.16.0 keras-tuner empyrical-reloaded spektral yfinance

In [ ]:
import os, sys
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/adam-909/4yp.git /content/repo
    else:
        !cd /content/repo && git pull
    os.chdir('/content/repo/4YP-main')
    RESULTS_BASE = "/content/drive/MyDrive/FINAL_RESULTS"
    from google.colab import drive
    drive.mount('/content/drive')
else:
    os.chdir('/home/adam/new4YP/4YP-main')
    RESULTS_BASE = "FINAL_RESULTS"

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from empyrical import sharpe_ratio, sortino_ratio, max_drawdown, annual_return, annual_volatility, calmar_ratio
import random, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
print(f"TensorFlow: {tf.__version__}")

## 2. Configuration

In [ ]:
# =============================
# PERTURBATION PARAMETERS
# =============================
PERTURBATION_MODE = "subtract"       # "add" or "subtract"
PERTURBATION_FRACTION = 0.5          # e.g. 0.25, 0.50, 0.75, 1.00
# =============================

EXPERIMENT_NAME = "6a_perturbation_gcn"
SEED = 40
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

TRAIN_START = 2011
TEST_START = 2017
TEST_END = 2023
VOL_TARGET = 0.15

# Rolling Pearson configuration (threshold=0.5 for ablation)
CORRELATION_LOOKBACK = 20
CORRELATION_THRESHOLD = 0.5
USE_EQUITY_RETURNS = True

# GCN hyperparameters (match lstm_gcn_rolling_pearson.ipynb baseline exactly)
TOTAL_TIME_STEPS = 20
TRAIN_VALID_RATIO = 0.8
NUM_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 25
HIDDEN_LAYER_SIZE = 10
GCN_UNITS = 16
DROPOUT_RATE = 0.4
LEARNING_RATE = 0.001
MAX_GRADIENT_NORM = 0.01
NUM_GCN_LAYERS = 2
BATCH_SIZE = 32

returns_type = "equity" if USE_EQUITY_RETURNS else "straddle"
CONFIG_NAME = f"{PERTURBATION_MODE}_{PERTURBATION_FRACTION:.2f}_lb{CORRELATION_LOOKBACK}_th{CORRELATION_THRESHOLD}_{returns_type}"

print(f"Experiment: {EXPERIMENT_NAME} (seed={SEED})")
print(f"Config: {CONFIG_NAME}")
print(f"Perturbation: mode={PERTURBATION_MODE}, fraction={PERTURBATION_FRACTION}")
print(f"Baseline: threshold={CORRELATION_THRESHOLD}, lookback={CORRELATION_LOOKBACK}, returns={returns_type}")

## 3. Helper Functions

In [ ]:
def calc_daily_returns(df, returns_col="captured_returns"):
    num_tickers = df["identifier"].nunique()
    daily_ret = df.groupby("time")[returns_col].sum() / num_tickers
    daily_ret.index = pd.to_datetime(daily_ret.index)
    return daily_ret.sort_index()

def calc_vol_scaled_returns(daily_returns, target_vol=0.15):
    v = daily_returns.std() * np.sqrt(252)
    return daily_returns * (target_vol / v) if v > 0 else daily_returns

def calc_metrics(dr, name="Strategy"):
    return {"Strategy": name, "E[Ret.]": annual_return(dr), "Vol.": annual_volatility(dr),
            "Sharpe": sharpe_ratio(dr), "Sortino": sortino_ratio(dr), "Max DD": -max_drawdown(dr),
            "Calmar": calmar_ratio(dr), "Hit Rate": (dr > 0).mean(),
            "Avg P/L": dr[dr > 0].mean() / abs(dr[dr < 0].mean()) if (dr < 0).any() else np.nan}

def calc_metrics_vol_normalized(dr, name, tv=0.15):
    s = calc_vol_scaled_returns(dr, tv)
    return calc_metrics(s, name + " (Vol-Norm)"), s

def display_metrics(m):
    df = pd.DataFrame([m]).set_index("Strategy")
    for c in ["E[Ret.]","Vol.","Max DD","Hit Rate"]:
        if c in df.columns: df[c] = df[c].apply(lambda x: f"{x:.2%}")
    for c in ["Sharpe","Sortino","Calmar","Avg P/L"]:
        if c in df.columns: df[c] = df[c].apply(lambda x: f"{x:.3f}")
    display(df)

def calc_yearly_sharpes(dr):
    return {y: sharpe_ratio(dr[dr.index.year == y]) for y in sorted(dr.index.year.unique())}

## 4. Data Loading (Rolling Pearson)

In [ ]:
features_path = "data/straddle_features/features.csv"
if 'google.colab' in str(get_ipython()):
    features_path = "/content/drive/MyDrive/features.csv"

df = pd.read_csv(features_path)
df["date"] = pd.to_datetime(df["date"])
print(f"Loaded {len(df)} rows, {df['ticker'].nunique()} tickers")

In [ ]:
from gml.graph_model_inputs import RollingGraphModelFeatures

rolling_features = RollingGraphModelFeatures(
    df=df, total_time_steps=TOTAL_TIME_STEPS,
    correlation_lookback=CORRELATION_LOOKBACK,
    correlation_threshold=CORRELATION_THRESHOLD,
    returns_column="daily_returns",
    use_equity_returns=USE_EQUITY_RETURNS,
    start_boundary=TRAIN_START, test_boundary=TEST_START, test_end=TEST_END,
    train_valid_ratio=TRAIN_VALID_RATIO,
    split_tickers_individually=True, time_features=False,
)

datasets = rolling_features.make_rolling_graph_dataset(sliding_window=True)
train_data = datasets["train"]
valid_data = datasets["valid"]
test_data  = datasets["test"]

print(f"\nBaseline shapes:")
print(f"  Train: inputs={train_data['inputs'].shape}, adj={train_data['adjacency'].shape}")
print(f"  Valid: inputs={valid_data['inputs'].shape}, adj={valid_data['adjacency'].shape}")
print(f"  Test:  inputs={test_data['inputs'].shape}, adj={test_data['adjacency'].shape}")

## 5. Inject Perturbation

In [ ]:
from gml.experiment_utils import perturb_adjacencies

# Save original adjacencies for comparison
train_adj_baseline = train_data["adjacency"].copy()
test_adj_baseline = test_data["adjacency"].copy()

print(f"Perturbing adjacencies (mode={PERTURBATION_MODE}, fraction={PERTURBATION_FRACTION})...")
train_data["adjacency"] = perturb_adjacencies(train_data["adjacency"], PERTURBATION_MODE, PERTURBATION_FRACTION, base_seed=42)
valid_data["adjacency"] = perturb_adjacencies(valid_data["adjacency"], PERTURBATION_MODE, PERTURBATION_FRACTION, base_seed=43)
test_data["adjacency"]  = perturb_adjacencies(test_data["adjacency"],  PERTURBATION_MODE, PERTURBATION_FRACTION, base_seed=44)

# Verify
def count_edges(adj):
    counts = []
    for A in adj:
        A = A.copy(); np.fill_diagonal(A, 0)
        counts.append(int((A > 0).sum() / 2))
    return np.array(counts)

baseline_edges = count_edges(test_adj_baseline)
perturbed_edges = count_edges(test_data["adjacency"])

if PERTURBATION_MODE == "subtract":
    expected = (1 - PERTURBATION_FRACTION) * baseline_edges
elif PERTURBATION_MODE == "add":
    expected = (1 + PERTURBATION_FRACTION) * baseline_edges

print(f"\nTest edge counts verification:")
print(f"  Baseline: mean={baseline_edges.mean():.1f}, range=[{baseline_edges.min()}, {baseline_edges.max()}]")
print(f"  Perturbed: mean={perturbed_edges.mean():.1f}, range=[{perturbed_edges.min()}, {perturbed_edges.max()}]")
print(f"  Expected (approx): mean={expected.mean():.1f}")
print(f"  Diff from expected: mean abs={np.abs(perturbed_edges - expected).mean():.2f}")

In [ ]:
# Heatmap comparison: baseline vs perturbed (averaged across test windows)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
avg_baseline = test_adj_baseline.mean(axis=0)
avg_perturbed = test_data["adjacency"].mean(axis=0)

sns.heatmap(avg_baseline, ax=axes[0], cmap="viridis", vmin=0)
axes[0].set_title("Baseline (Pearson) — Average Adjacency")

sns.heatmap(avg_perturbed, ax=axes[1], cmap="viridis", vmin=0)
axes[1].set_title(f"Perturbed ({PERTURBATION_MODE} {PERTURBATION_FRACTION}) — Average Adjacency")

plt.tight_layout(); plt.show()

## 6. Model Definition (GCN)

In [ ]:
from gml.graph_model_2 import DynamicGraphConvolution, GraphSharpeLoss

def build_rolling_lstm_gcn_model(num_tickers, time_steps, input_size,
                                  hidden_layer_size=10, gcn_units=16,
                                  dropout_rate=0.4, learning_rate=0.001,
                                  max_gradient_norm=0.01, num_gcn_layers=2):
    feature_input = keras.Input(shape=(num_tickers, time_steps, input_size), name="features")
    adjacency_input = keras.Input(shape=(num_tickers, num_tickers), name="adjacency")

    shared_lstm = layers.LSTM(hidden_layer_size, return_sequences=True, dropout=dropout_rate,
                              activation="tanh", recurrent_activation="sigmoid", name="shared_lstm")
    lstm_outputs = []
    for i in range(num_tickers):
        ticker_slice = layers.Lambda(lambda x, idx=i: x[:, idx, :, :])(feature_input)
        lstm_outputs.append(shared_lstm(ticker_slice))
    stacked_lstm = layers.Lambda(lambda t: tf.stack(t, axis=2))(lstm_outputs)

    gcn_output = DynamicGraphConvolution(units=gcn_units)([stacked_lstm, adjacency_input])
    gcn_output = layers.ReLU()(gcn_output)
    if num_gcn_layers == 2:
        gcn_output = DynamicGraphConvolution(units=gcn_units)([gcn_output, adjacency_input])
        gcn_output = layers.ReLU()(gcn_output)

    residual = layers.TimeDistributed(layers.TimeDistributed(
        keras.layers.Dense(gcn_units, activation="linear")))(stacked_lstm)
    x = layers.Add()([gcn_output, residual])

    output = layers.TimeDistributed(layers.TimeDistributed(
        keras.layers.Dense(1, activation=tf.nn.tanh,
                          kernel_constraint=keras.constraints.max_norm(3))))(x)
    output = layers.Permute((2, 1, 3))(output)

    model = keras.Model(inputs=[feature_input, adjacency_input], outputs=output)
    adam = keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=max_gradient_norm)
    model.compile(loss=GraphSharpeLoss(output_size=1), optimizer=adam)
    return model

num_tickers = train_data["inputs"].shape[1]
time_steps  = train_data["inputs"].shape[2]
input_size  = train_data["inputs"].shape[3]

print(f"Building GCN with perturbed adjacencies:")
print(f"  num_tickers={num_tickers}, time_steps={time_steps}, input_size={input_size}")

model = build_rolling_lstm_gcn_model(
    num_tickers=num_tickers, time_steps=time_steps, input_size=input_size,
    hidden_layer_size=HIDDEN_LAYER_SIZE, gcn_units=GCN_UNITS,
    dropout_rate=DROPOUT_RATE, learning_rate=LEARNING_RATE,
    max_gradient_norm=MAX_GRADIENT_NORM, num_gcn_layers=NUM_GCN_LAYERS,
)
print(f"\nTotal parameters: {model.count_params():,}")

## 7. Training

In [ ]:
X_train = [train_data["inputs"], train_data["adjacency"]]
y_train = train_data["outputs"]
w_train = train_data["active_entries"]

X_valid = [valid_data["inputs"], valid_data["adjacency"]]
y_valid = valid_data["outputs"]
w_valid = valid_data["active_entries"]

print(f"Train: {train_data['inputs'].shape[0]} samples, Valid: {valid_data['inputs'].shape[0]} samples")

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True, verbose=1)

print("=" * 60)
print(f"Training {EXPERIMENT_NAME} ({CONFIG_NAME}, seed={SEED})")
print(f"  Perturbation: {PERTURBATION_MODE} {PERTURBATION_FRACTION}")
print("=" * 60)

history = model.fit(
    X_train, y_train, sample_weight=w_train,
    validation_data=(X_valid, y_valid, w_valid),
    epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[early_stopping], verbose=1,
)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Val")
plt.xlabel("Epoch"); plt.ylabel("Loss (Neg Sharpe)")
plt.title(f"{EXPERIMENT_NAME}: {PERTURBATION_MODE} {PERTURBATION_FRACTION} Training Loss")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f"Epochs: {len(history.history['loss'])}, Best val: {min(history.history['val_loss']):.4f}")

## 8. Evaluation

In [ ]:
X_test = [test_data["inputs"], test_data["adjacency"]]
predictions = model.predict(X_test)
print(f"Predictions shape: {predictions.shape}")

In [ ]:
positions = predictions[:, :, -1, 0].reshape(-1)
returns   = test_data["outputs"][:, :, -1, 0].reshape(-1)
captured  = positions * returns
dates     = test_data["date"][:, :, -1, 0].reshape(-1)
ids       = test_data["identifier"][:, :, -1, 0].reshape(-1)

results_df = pd.DataFrame({"time": dates, "identifier": ids,
                           "position": positions, "returns": returns,
                           "captured_returns": captured})
results_df["time"] = pd.to_datetime(results_df["time"])
results_df = results_df[results_df["identifier"] != "0"]
print(f"Results: {len(results_df)} rows")

In [ ]:
daily_returns = calc_daily_returns(results_df)

print("\n" + "=" * 60)
print(f"{EXPERIMENT_NAME} ({CONFIG_NAME}) — Results")
print("=" * 60)
metrics_raw = calc_metrics(daily_returns, EXPERIMENT_NAME)
display_metrics(metrics_raw)

metrics_norm, _ = calc_metrics_vol_normalized(daily_returns, EXPERIMENT_NAME, VOL_TARGET)
display_metrics(metrics_norm)

yearly_sharpes = calc_yearly_sharpes(daily_returns)
print("\nYearly Sharpes:")
for y, s in yearly_sharpes.items():
    print(f"  {y}: {s:.4f}")

## 9. Save Results

In [ ]:
from gml.experiment_utils import save_experiment_results, compute_graph_stats

test_dates_arr = pd.to_datetime(test_data["date"][:, 0, -1, 0])
graph_stats = compute_graph_stats(test_data["adjacency"], threshold=0.0)

hyperparams = {
    "model_type": "GCN_rolling_pearson_perturbed",
    "hidden_layer_size": HIDDEN_LAYER_SIZE,
    "gcn_units": GCN_UNITS,
    "num_gcn_layers": NUM_GCN_LAYERS,
    "dropout_rate": DROPOUT_RATE,
    "learning_rate": LEARNING_RATE,
    "max_gradient_norm": MAX_GRADIENT_NORM,
    "batch_size": BATCH_SIZE,
    "correlation_lookback": CORRELATION_LOOKBACK,
    "correlation_threshold": CORRELATION_THRESHOLD,
    "use_equity_returns": USE_EQUITY_RETURNS,
    "perturbation_mode": PERTURBATION_MODE,
    "perturbation_fraction": PERTURBATION_FRACTION,
    "total_time_steps": TOTAL_TIME_STEPS,
    "train_start": TRAIN_START, "test_start": TEST_START, "test_end": TEST_END,
}

save_experiment_results(
    experiment_name=EXPERIMENT_NAME, seed=SEED, config_name=CONFIG_NAME,
    predictions=predictions, results_df=results_df,
    daily_returns=daily_returns, metrics_raw=metrics_raw,
    metrics_norm=metrics_norm, yearly_sharpes=yearly_sharpes,
    training_history=history.history, hyperparams=hyperparams,
    test_dates=test_dates_arr.values,
    adjacency=test_data["adjacency"],
    graph_stats=graph_stats,
    model=model,
    base_dir=RESULTS_BASE,
)

## 10. Summary

In [ ]:
print("=" * 60)
print(f"EXPERIMENT: {EXPERIMENT_NAME}")
print(f"Config: {CONFIG_NAME}")
print("=" * 60)
print(f"Perturbation: {PERTURBATION_MODE} {PERTURBATION_FRACTION}")
print(f"Baseline edges (test mean): {baseline_edges.mean():.1f}")
print(f"Perturbed edges (test mean): {perturbed_edges.mean():.1f}")
print(f"\nSharpe: {metrics_raw['Sharpe']:.3f}")
print(f"Return: {metrics_raw['E[Ret.]']:.2%}, Vol: {metrics_raw['Vol.']:.2%}")
print(f"Saved to: {RESULTS_BASE}/{EXPERIMENT_NAME}/{CONFIG_NAME}/seed_{SEED}/")